# Instationäre Behälterleerung – numerische Lösung

**BIW3-10 Weiterführende Hydromechanik · Übung 3 · Aufgabe 5**

In der Übung haben wir die Behälterleerung analytisch gelöst (Trennung der Variablen, Folie 26):

$$A\,\frac{\mathrm{d}h}{\mathrm{d}t} = -\alpha A_{\mathrm{o}}\sqrt{2gh} \qquad\Longrightarrow\qquad h(t) = h_0\left(1-\frac{t}{T_E}\right)^2,\qquad T_E = \frac{2A\sqrt{h_0}}{\alpha A_{\mathrm{o}}\sqrt{2g}}$$

**Was hier passiert:** Wir lösen dieselbe DGL *numerisch* mit dem **expliziten Euler-Verfahren** und vergleichen mit der exakten Lösung. Dabei lernst du:

- wie ein numerisches Verfahren eine DGL „schrittweise“ löst,
- wie der Fehler von der Schrittweite $\Delta t$ abhängt (Aufgabe 5b),
- warum die Schrittweite physikalisch genau dem **quasi-stationären „Momentaufnahmen“-Bild** der Vorlesung entspricht (Aufgabe 5c).

## 1. Parameter (wie in Aufgabe 4 des Übungsblatts)

Zylinderspeicher mit scharfkantiger Bodenöffnung, kein Zulauf. Alle Werte in **SI-Einheiten** (Meter, Sekunden) – so wie die Formeln es erwarten.

In [ ]:
# ======================================================================
# 1. PARAMETER DER AUFGABE 4 (Übungsblatt) – alle in SI-Einheiten!
# ======================================================================
# Jede Zeile: Name = Wert   # Einheit | Bedeutung

g     = 9.81    # m/s^2  | Erdbeschleunigung - sie "zieht" das Wasser nach unten
                #          und treibt über die Druckhöhe h die Ausflussgeschwindigkeit.

D     = 2.0     # m      | Behälter-Innendurchmesser (Zylinder).
                #          Dient nur zur Berechnung des Querschnitts A.

d_o   = 0.08    # m      | Durchmesser der scharfkantigen Bodenöffnung.
                #          Achtung: Blattangabe war 80 mm -> hier in Meter umgerechnet!
                #          Faustregel fürs Rechnen: immer SI-Einheiten verwenden.

alpha = 0.62    # -      | Ausflusszahl (dimensionslos), alpha = 1/sqrt(1 + zeta_o).
                #          Sie "schmiert" die Realität ein: Strahlverengung und
                #          Verluste an der scharfkantigen Öffnung verkleinern den
                #          idealen Torricelli-Abfluss um den Faktor alpha (Folie 16).

h0    = 3.6     # m      | Anfangswasserstand ÜBER der Öffnung (Beginn t = 0).
                #          Der Spiegel h(t) ist unsere unbekannte Funktion!

In [ ]:
# ======================================================================
# 2. ABGELEITETE GRÖSSEN - werden aus den Parametern berechnet
# ======================================================================
import numpy as np                 # Numerik-Bibliothek (Arrays, sqrt, pi, ...)
import matplotlib.pyplot as plt    # Plot-Bibliothek

# --- Flächen (Kreisfläche A = pi * d^2 / 4) ---------------------------
A   = np.pi * D**2 / 4    # m^2 | Behälterquerschnitt (durch D bestimmt)
A_o = np.pi * d_o**2 / 4  # m^2 | Öffnungsquerschnitt (durch d_o bestimmt)

# --- Kappa: der Koeffizient VOR der Wurzel in der DGL -----------------
#   Ausgangs-DGL (Volumenbilanz + Torricelli, Aufgabe 4b):
#       A * dh/dt = -alpha * A_o * sqrt(2*g*h)
#   Durch A teilen:
#       dh/dt = -[alpha*A_o*sqrt(2g)/A] * sqrt(h) = -kappa*sqrt(h)
kappa = alpha * A_o * np.sqrt(2*g) / A    # m^0.5/s  (Einheiten genau prüfen!)

# --- Analytische Auslaufzeit (Übungsblatt, Aufgabe 4c) ----------------
#   T_E = 2*A*sqrt(h0) / (alpha*A_o*sqrt(2g))
#   Bedeutung: Zeitpunkt, an dem h(t) = 0 ist (Behälter "leer" bis Öffnungsniveau).
T_E = 2 * A * np.sqrt(h0) / (alpha * A_o * np.sqrt(2*g))    # s

# --- Anfangsabfluss: Torricelli mit dem ANFÄNGLICHEN Wasserstand ------
#   Q0 = alpha*A_o*sqrt(2*g*h0)   -> der größte Abfluss im ganzen Vorgang.
Q0 = alpha * A_o * np.sqrt(2*g*h0)    # m^3/s

# --- Kontrolle: sind das die Werte aus dem Übungsblatt? ---------------
print(f"A         = {A:.4f} m^2        (Blatt: 3,142 m^2)")
print(f"A_o       = {A_o*1e3:.3f} e-3 m^2   (Blatt: 5,03 e-3 m^2)")
print(f"kappa     = {kappa:.6e} m^0.5/s")
print(f"T_E       = {T_E:.1f} s = {T_E/60:.2f} min   (Blatt: 863,6 s = 14,4 min)")
print(f"Q_0       = {Q0*1000:.1f} L/s       (Blatt: 26,2 L/s)")
print(f"A_o/A     = {A_o/A:.2e}  -> v_Spiegel ist nur {A_o/A*100:.2f} % von v_Ausfluss")

## 2. Analytische Lösung als Referenz

Aus der Übung (Aufgabe 4c, Trennung der Variablen):

- Spiegel: $h(t) = h_0\,(1-t/T_E)^2$ – eine **Parabel** (steil am Anfang, flach am Ende),
- Abfluss: $Q(t) = \alpha A_{\mathrm{o}}\sqrt{2g\,h(t)}$ – **linearer Abfall** von $Q_0$ auf 0.

In [ ]:
# ======================================================================
# 3. ANALYTISCHE LÖSUNG (aus der Übung) - unser Referenz-/Vergleichswert
# ======================================================================
# Beide Funktionen nehmen die Zeit t [s] entgegen und liefern [m] bzw. [m^3/s].

def h_analytisch(t):
    """Exakter Spiegelverlauf h(t) [m] zur Zeit t [s].
    Herkunft: Aufgabe 4c - Trennung der Variablen der Behälter-DGL."""
    return h0 * (1 - t/T_E)**2        # Parabel: bei t=T_E genau 0

def Q_analytisch(t):
    """Abfluss Q(t) [m^3/s] zur Zeit t [s] (Torricelli mit momentanem h).
    Linearer Abfall, weil sqrt(h) linear in t fällt."""
    h = h_analytisch(t)
    # np.maximum(h, 0) = "Klemme auf min. 0": verhindert Wurzeln aus
    # negativen Werten, falls t zufällig über T_E hinausgeht (Rundung!).
    return alpha * A_o * np.sqrt(2*g*np.maximum(h, 0.0))

# --- Plotten -----------------------------------------------------------
# Wir erzeugen 400 Zeitpunkte von 0 bis T_E und zeichnen beide Kurven.
t = np.linspace(0, T_E, 400)          # 400 gleichmäßige Zeitpunkte [s]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))   # 2 Plots nebeneinander

ax[0].plot(t, h_analytisch(t), 'b')
ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('h(t) [m]')
ax[0].set_title('Spiegel h(t): Parabel')
ax[0].grid(alpha=0.3)

ax[1].plot(t, Q_analytisch(t)*1000, 'r')          # *1000: m^3/s -> L/s
ax[1].set_xlabel('t [s]'); ax[1].set_ylabel('Q(t) [L/s]')
ax[1].set_title('Abfluss Q(t): linearer Abfall')
ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Beobachtung: h fällt anfangs SCHNELL (großes -dh/dt, weil viel Wasser drückt),
# am Ende kaum noch - die letzten 25 % Volumen brauchen die halbe Zeit (Folie 21).

## 3. Explizites Euler-Verfahren (Aufgabe 5a – **hier setzt du an!**)

**Idee:** Wir kennen den Anfangsstand $h_0$. Die DGL liefert zu jedem Zustand die **Steigung** $\mathrm{d}h/\mathrm{d}t = f(t,h)$. Das Verfahren geht dann schrittweise geradlinig weiter:

$$h_{n+1} = h_n + \Delta t \cdot f(t_n, h_n), \qquad f(t,h) = -\kappa\sqrt{h}$$

Anschaulich: In jedem Schritt wird das Problem als *quasi-stationäres Momentbild* behandelt (Folie 26) – die Schrittweite $\Delta t$ ist die Länge eines Momentbildes. **Kleines $\Delta t$ = Spiegel ändert sich pro Bild kaum = Quasistationarität gut erfüllt.**

In [ ]:
# ======================================================================
# 4. EXPLIZITES EULER-VERFAHREN   (Aufgabe 5a - HIER SETZT DU AN!)
# ======================================================================
# Schema des Verfahrens (vorwärts/explizit):
#
#       h[n+1] = h[n] + dt * f(t[n], h[n])
#
#   mit der rechten Seite (Steigung) der DGL:
#       f(t, h) = -kappa * sqrt(h)      [m/s]
#
#   Ablauf pro Schritt:
#       (1) Steigung am aktuellen Punkt ausrechnen
#       (2) einen kleinen geradlinigen Schritt der Länge dt*Steigung machen
#       (3) Zeit um dt weiterstellen
#       (4) stoppen, sobald der Behälter leer ist (h <= 0)

def euler(dt, t_max=None):
    """Löst  dh/dt = -kappa*sqrt(h)  mit dem expliziten Euler-Verfahren.

    Parameter
    ---------
    dt    : float
            Zeitschrittweite [s]. Klein = genauer, aber mehr Rechenschritte.
    t_max : float oder None
            Sicherheitsgrenze für die Gesamtzeit [s].
            None heißt: 3*T_E (der Vorgang sollte bei T_E enden).

    Rückgabe
    --------
    t : numpy-Array  Zeitpunkte [s],  t[0] = 0
    h : numpy-Array  Wasserstände [m], h[0] = h0
        (beide gleich lang, h[-1] <= 0 markiert das leere Behälterende)
    """
    # --- Sicherheitsgrenze, damit die Schleife garantiert endet ---------
    if t_max is None:
        t_max = 3 * T_E
    n_max = int(t_max / dt) + 2        # maximale Anzahl Schritte

    # --- Speicher vorab anlegen + ANFANGSBEDINGUNG ----------------------
    t = np.zeros(n_max + 1)            # Zeitpunkte [s]
    h = np.zeros(n_max + 1)            # Wasserstände [m]
    h[0] = h0                          # Anfangsbedingung: h(0) = h0  <-- fertig vorgegeben

    # --- Zeitschleife: hier passiert das eigentliche Verfahren ----------
    for n in range(n_max):
        # (1) Steigung f(t_n, h_n) aus der DGL bestimmen.
        #     max(h[n], 0.0) klemmt h auf min. 0: sqrt(negativ) wäre ein
        #     Fehler. Genau an dieser Stelle sieht man später im Zoom (Abschn. 8),
        #     warum grobe Schrittweiten h unter 0 drücken wollen!
        f = -kappa * np.sqrt(max(h[n], 0.0))       # Steigung [m/s]  <-- fertig vorgegeben

        # (2) TODO: EULER-SCHRITT.
        #     Neuer Stand = alter Stand + dt * Steigung:
        #         h[n+1] = ...

        # (3) TODO: Zeit um einen Schritt weiterstellen:
        #         t[n+1] = ...

        # (4) TODO: Abbruchbedingung "Behälter leer".
        #     Wenn h[n+1] <= 0:
        #         - h[n+1] auf 0 klemmen:  h[n+1] = max(h[n+1], 0.0)
        #           (physikalisch: unterhalb der Öffnung fließt nichts mehr,
        #            negative Wasserstände gibt es nicht - Folie 18)
        #         - Ergebnis zurückgeben:  return t[:n+2], h[:n+2]

        raise NotImplementedError(
            "Aufgabe 5a: Erst die TODOs in euler() umsetzen (Schritte 2-4)!")

    return t, h   # (sollte normalerweise nicht erreicht werden - siehe (4))

In [ ]:
# ======================================================================
# 5. ERSTER TEST: einmal mit dt = 10 s durchrechnen
# ======================================================================
# Wir rufen euler() auf und vergleichen die NUMERISCHE Auslaufzeit
# (letzter Zeitpunkt im Array, dort ist h <= 0) mit dem ANALYTISCHEN
# Wert T_E aus dem Übungsblatt (863,6 s = 14,4 min).
# Der relative Fehler in Prozent:
#
#       Fehler = (T_E,num - T_E) / T_E * 100 %
#
# Negativ = Euler entleert ZU SCHNELL (das wird gleich systematisch so sein).

t_num, h_num = euler(dt=10.0)      # <-- Funktion aus Abschnitt 4
T_E_num = t_num[-1]                # letzter Zeitpunkt = Ende der Rechnung

print(f"T_E,num  (dt = 10 s)   = {T_E_num:.1f} s")
print(f"T_E,analytisch        = {T_E:.1f} s")
print(f"Fehler                = {(T_E_num - T_E)/T_E*100:+.2f} %")
print(f"Anzahl Schritte       = {len(t_num)-1}")

In [ ]:
# ======================================================================
# 6. VERGLEICHSPLOT: numerisch vs. analytisch
# ======================================================================
# Die blauen Punkte (Euler) sollten auf der blauen Kurve (exakt) liegen.
# Achte besonders auf das ENDE der Kurve - dort sieht man den Fehler zuerst.

t_num, h_num = euler(dt=10.0)
t_ref = np.linspace(0, T_E, 400)               # feine Kurve für die Exakte

plt.figure(figsize=(8, 4))
plt.plot(t_ref, h_analytisch(t_ref), 'b-', linewidth=2, label='analytisch (exakt)')
plt.plot(t_num, h_num, 'ro-', markersize=3, label='Euler (dt = 10 s)')
plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.legend(); plt.grid(alpha=0.3)
plt.title('Behälterleerung: Euler vs. analytisch')
plt.show()

# Beobachtung: Die Euler-Punkte liegen leicht UNTER der exakten Kurve.
# Grund: h(t) ist konvex (nach oben gekrümmt) - Tangenten an eine konvexe
# Kurve liegen UNTER der Kurve. Euler folgt aber genau Tangenten!

## 4. Schrittweitenstudie (Aufgabe 5b)

Wiederhole die Rechnung für $\Delta t = 100$ s, $10$ s, $1$ s, $0{,}1$ s und trage die numerische Auslaufzeit samt Fehler gegenüber dem analytischen Wert ($T_E = 863{,}6$ s) zusammen.

**Frage:** Halbiert sich der Fehler, wenn man die Schrittweite halbiert? (Tipp: Euler ist ein Verfahren 1. Ordnung …)

In [ ]:
# ======================================================================
# 7. SCHRITTWEITENSTUDIE (Aufgabe 5b)
# ======================================================================
# Wir laufen über eine Liste von Schrittweiten und geben für jede:
#   - die numerische Auslaufzeit T_E,num  (= letzter Zeitpunkt)
#   - den relativen Fehler gegenüber der analytischen Lösung
#
# Erwartung: je KLEINER dt, desto KLEINER der Fehler.
# Struktur der Ausgabe: dt | T_E,num | Fehler in %

print(f"{'dt [s]':>8} | {'T_E,num [s]':>12} | {'Fehler [%]':>10}")
print('-' * 42)

for dt in [100.0, 10.0, 1.0, 0.1]:        # vier verschiedene Schrittweiten
    t_num, h_num = euler(dt)              # Rechnung mit dieser Schrittweite
    T_E_num = t_num[-1]                   # letzter Zeitpunkt = Auslaufende
    fehler = (T_E_num - T_E) / T_E * 100  # relativer Fehler in Prozent
    print(f"{dt:8.1f} | {T_E_num:12.1f} | {fehler:+10.2f}")

# Beobachtung: Der Fehler ist systematisch NEGATIV (zu schnelle Entleerung)
# und schrumpft ungefähr proportional zu dt - das ist die Signatur eines
# Verfahrens 1. Ordnung: dt/10 -> Fehler ~ 1/10.

In [ ]:
# ======================================================================
# 8. ZOOM ANS ENDE: Was passiert nahe h -> 0 bei grobem dt?
# ======================================================================
# Hier sieht man den Euler-Fehler am deutlichsten:
#   Die exakte Kurve läuft flach in h = 0 aus (Wurzel-Funktion!).
#   Euler dagegen nimmt am letzten Punkt noch eine STEILE Tangente,
#   springt bei grobem dt unter 0 - und würde ohne "Klemme" negative
#   Wasserstände berechnen (physikalischer Unsinn!).

plt.figure(figsize=(8, 4))
t_ref = np.linspace(0.75*T_E, 1.05*T_E, 200)                 # Zeitfenster ans Ende
plt.plot(t_ref, h_analytisch(np.minimum(t_ref, T_E)), 'b-',
         linewidth=2, label='analytisch')                    # min(t,T_E): nach T_E bleibt 0

for dt, farbe in [(100.0, 'orange'), (10.0, 'green')]:
    t_num, h_num = euler(dt)
    plt.plot(t_num, h_num, 'o-', color=farbe, markersize=5,
             label=f'Euler (dt = {dt:g} s)')

plt.xlabel('t [s]'); plt.ylabel('h [m]')
plt.xlim(0.75*T_E, 1.05*T_E)
plt.ylim(-0.15, 0.4)                                          # negatives h sichtbar machen
plt.legend(); plt.grid(alpha=0.3)
plt.title('Nahe h -> 0: Euler schneidet die Parabel ab')
plt.show()

# Deutung: f(h) = -kappa*sqrt(h) hat bei h -> 0 eine UNENDLICHE Steigung
# (Ableitung von sqrt). Grobe Schrittweiten "überspringen" das flache
# Auslaufen daher komplett. Physik dahinter: der Abfluss Q ~ sqrt(h)
# wird zwar klein, ABER der Absenkungsgradient wird groß -klassischer Fall
# für steife ("stiff") Gleichungen am Ende.

## 5. Vergleich mit adaptiver Methode (scipy)

Das Euler-Verfahren nutzt eine **feste** Schrittweite. Professionelle Löser wie `scipy.integrate.solve_ivp` passen die Schrittweite **automatisch** an: groß, wo sich wenig tut; klein, wo es steil wird (Runge-Kutta 4/5). Wir nutzen es hier als unabhängigen Referenzwert.

In [ ]:
# ======================================================================
# 9. ADAPTIVER VERGLEICH: scipy.integrate.solve_ivp (RK45)
# ======================================================================
# solve_ivp braucht:
#   rhs(t, y)     : rechte Seite der DGL. y ist ein VEKTOR [h] - deshalb y[0].
#   t_span        : Zeitintervall (Anfang, Ende) [s]
#   [h0]          : Anfangszustand (ebenfalls Vektor!)
#   events        : Funktion y[0]=0 markiert das Ereignis "Behälter leer"
#   rtol, atol    : relative/absolute Toleranz -> Genauigkeit der Adaption

from scipy.integrate import solve_ivp

def rhs(t, y):
    """Rechte Seite der DGL: dh/dt = -kappa*sqrt(h).
    y[0] ist der Wasserstand h (Vektor, damit später auch
    Systeme mit mehreren Unbekannten möglich wären)."""
    return [-kappa * np.sqrt(max(y[0], 0.0))]

# Event: nullstelle von y[0] = Wasserstand -> "Behälter leer"
def event_leer(t, y):
    return y[0]                    # wird zu 0, genau wenn h = 0
event_leer.terminal = True          # Rechnung BEI dem Event anhalten
event_leer.direction = -1           # nur Durchgang von + nach - zählen

t_span = (0.0, 3*T_E)               # von 0 bis 3*T_E s (reichlich Luft)
sol = solve_ivp(rhs, t_span, [h0],
                events=event_leer, rtol=1e-8, atol=1e-10)

# sol.t_events[0][0] = Zeitpunkt, an dem das Event (h = 0) ausgelöst hat:
T_E_adaptiv = sol.t_events[0][0]

print(f"T_E (solve_ivp, RK45) = {T_E_adaptiv:.2f} s")
print(f"T_E (analytisch)      = {T_E:.2f} s")
print(f"Abweichung            = {(T_E_adaptiv-T_E)/T_E*100:+.4f} %")

# Interpretation: Mit ADAPTIVER Schrittweite ist die Abweichung extrem klein.
# Der Löser verkleinert dt automatisch, wo die Kurve steil/krümmungsreich ist
# (hier: am Ende nahe h -> 0). Genau das kann festes Euler nicht.

## 6. Interpretation (Aufgabe 5c) – in eigenen Worten beantworten

1. **Was entspricht die Schrittweite $\Delta t$ im quasi-stationären Bild der Vorlesung?** *(Tipp: Was bedeutet es, jeden Schritt als „Momentaufnahme“ zu rechnen – Folie 26?)*
2. **Warum entleert die Euler-Lösung systematisch zu schnell?** *(Ist $h(t)$ konvex oder konkav? Wo liegen Tangenten an eine konvexe Kurve?)*
3. **Warum scheitert Euler nahe $h \to 0$ bei grobem $\Delta t$?** *(Was macht die Steigung $f(h) = -\kappa\sqrt{h}$ in der Nähe von $h=0$?)*
4. **Übertrage das Kriterium „kleines $\Delta t$“ auf die Vorlesungsfrage:** *Wann darf man quasi-stationär rechnen?* (Folie 26/36: „die Änderung muss lang sein gegen die charakteristische Zeit des Systems – und man muss sagen, welche.“)